# Modeling and Associations

### Imports

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf


In [3]:
cwd = Path.cwd()
root = next(p for p in [cwd, *cwd.parents] if (p / "data" / "raw" / "survey_data.csv").exists())
df_clean = pd.read_csv(root / "data" / "clean" / "clean_data.csv")

## Prepare Ordinal Variables

Since "Unsure / Don't Know" answers are coded as 5, all instances of 5 will be excluded from the following analyses since it does not fit on the high-end of the ordinal scale. 

In [7]:
# Make a copy of the cleaned data
num = df_clean.copy()

# Codes 1-5 remain in codebook direction. Code 5 is excluded as Don't know/Unsure.
ordinal = ["q1", "q2", "q3", "q5"]
# Convert ordinal columns to numeric and exclude code 5 (DK/Unsure)
for q in ordinal:
    num[q] = pd.to_numeric(num[q], errors="raise").astype(float)
    num[q] = num[q].where(num[q].between(1, 4))

# Ensure string type on categorical columns
for c in ["location", "sex", "q4"]:
    num[c] = num[c].astype(str)

print("Missing values after excluding code 5:")
print(num[ordinal].isna().sum())

Missing values after excluding code 5:
q1    12
q2    10
q3     1
q5    24
dtype: int64


We see above that our data cleaning has excluded between `1` and `24` respondents. This should be noted in the methodology section. Those that answered `5` may systematically differ from respondents that did not, possibly biasing results.

## Categorical Associations

Since our numeric data is `ordinal`, a simple Pearson's correlation coefficient would not work, since it requires the data to be continuous. Instead, we should turn to `rank-based correlations` (spearman) and `nonparametric` statistical inference tests like the `Mann-Whitney U Test`, which does not require any distributional assumptions! You can think of this test as a regular `t-test` without the normality assumption!

Below is a quick helper function calculating Cramer's V, a really helpful metric when working with survey data! In this example, it measures the relationship strength between `sex` and `awareness` (Q1-3) or even `location` and `likelihood to donate` (Q5). 

In [8]:
def cramers_v(ct):
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    return np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))

In [ ]:
rows = []
# For each categorical group we're interested in
for group in ["location", "sex"]:
    # For each question within those groups
    for question in ["q1", "q2", "q3", "q4", "q5"]:
        # Drop missing values for analysis
        sub = num[[group, question]].dropna()
        # Crosstab of group and question
        ct = pd.crosstab(sub[group], sub[question])
        # Define the metrics we're interested in
        row = {"group": group, "question": question, "n": len(sub), "chi2_p": stats.chi2_contingency(ct)[1], "cramers_v": cramers_v(ct)}
        if question != "q4":
            groups = [values.to_numpy(dtype=float) for _, values in sub.groupby(group)[question]]
            if len(groups) == 2:
                # Mann-Whitney U test for two independent samples (Ex. Does awareness of Green Machine significantly differ by sex?)
                u = stats.mannwhitneyu(groups[0], groups[1])
                # P-value (alpha = 0.05)
                row["mw_p"] = u.pvalue
        rows.append(row)

results = pd.DataFrame(rows)
# Correct for multiple comparisons using Holm-Bonferroni method
results["p_holm"] = multipletests(results["chi2_p"], method="holm")[1]
display(results.round(3))

,group,question,n,chi2_p,cramers_v,mw_p,prob_superiority,p_holm
0,location,q1,88,0.000,0.508,0.000,0.229,0.000
1,location,q2,90,0.081,0.273,0.031,0.366,0.406
2,location,q3,99,0.317,0.189,0.329,0.445,0.634
3,location,q4,100,0.002,0.410,NaN,NaN,0.017
4,location,q5,76,0.151,0.264,0.169,0.402,0.604
5,sex,q1,88,0.155,0.244,0.124,0.403,0.604
6,sex,q2,90,0.031,0.315,0.008,0.336,0.214
7,sex,q3,99,0.563,0.144,0.173,0.424,0.634
8,sex,q4,100,0.001,0.437,NaN,NaN,0.007
9,sex,q5,76,0.060,0.312,0.048,0.367,0.359


### Key results above ...

- Row 0. IRA awareness distributions significantly differ between **locations** (CO and MT) (_cramers v_ = 0.508, _p_ < .0001)

- Row 3. Importance of environmental issues significantly differ between **locations** (_cramers v_ = 0.410, _p_ = .017)

- Row 8. Importance of environmental issues significantly differ between **sexes** (_cramers v_ = 0.437, _p_ = .007)

No other significant distribution differences were recognized. Please note that this may be due to insufficient sample sizes for detecting differences. P-values are not facts, but merely evidence-backed suggestions!






## Logistic Models of Top-2 Responses

The code below runs a `logistic regression` model for each group/question combination, including a `location x sex` interaction term to control for potential confounding. For example, locations may differ in donations, but this may simply be a shadow of difference sex distributions in each state. 

In [10]:
results = []
for q in ["q1", "q2", "q3", "q5"]:
    d = num[[q, "location", "sex"]].dropna().copy()
    # Check scale direction: le(2) if 1 is best/highest, ge(3) if 4 is highest
    d["top2"] = d[q].le(2).astype(int)
    
    model = smf.logit("top2 ~ C(location) + C(sex)", d).fit(disp=0)
    ci = np.exp(model.conf_int())
    
    table = pd.DataFrame({
        "question": q,
        "n": len(d),
        "OR": np.exp(model.params),
        "CI_low": ci[0],
        "CI_high": ci[1],
        "p": model.pvalues
    }).drop("Intercept")
    
    results.append(table)

summary = pd.concat(results)
summary["p_holm"] = multipletests(summary["p"], method="holm")[1]
display(summary.round(3))


,question,n,OR,CI_low,CI_high,p,p_holm
C(location)[T.MT],q1,88,0.147,0.051,0.426,0.000,0.003
C(sex)[T.M],q1,88,0.861,0.315,2.349,0.770,1.000
C(location)[T.MT],q2,90,0.721,0.283,1.836,0.493,1.000
C(sex)[T.M],q2,90,0.356,0.141,0.899,0.029,0.202
C(location)[T.MT],q3,99,1.290,0.468,3.553,0.622,1.000
C(sex)[T.M],q3,99,0.557,0.210,1.478,0.240,1.000
C(location)[T.MT],q5,76,0.666,0.212,2.085,0.485,1.000
C(sex)[T.M],q5,76,0.300,0.101,0.892,0.030,0.202


#### The results above show a couple things:

1. Montana respondents are less likely to be aware of the IRA, compared to CO, regardless of sex. 
2. There is strong evidence that men are less likely to be willing to donate to an environmental organization compared to women, regardless of location.

## Spearman Associations with Donation Likelihood

The code below runs rank-based spearman correlations between questions 1-3 responses (awareness) and q5 responses (donation)

In [11]:
rows = []
for q in ["q1", "q2", "q3"]:
    d = num[[q, "q5"]].dropna()
    rho, p = stats.spearmanr(d[q].to_numpy(float), d["q5"].to_numpy(float))
    rows.append({"awareness": q, "n": len(d), "rho": rho, "p": p})
display(pd.DataFrame(rows).round(3))

,awareness,n,rho,p
0,q1,65,0.233,0.061
1,q2,70,0.336,0.004
2,q3,75,0.443,0.000


Results above confirm that awareness measures (q1-3) are associated with willingness to donate. This is not to say that people donate because they are more aware of environmental situations, but it does show strong associations between the two.  